In [ ]:
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score
from scipy.stats import pearsonr

from transformers import (
    AutoModelForSequenceClassification,
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    DistilBertModel,
    DistilBertPreTrainedModel,
    DistilBertConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)



In [ ]:
# =========================
# 1. GOOGLE DRIVE
# =========================
from google.colab import drive
drive.mount('/content/drive')

LOG_FILE_STEP1 = "/content/drive/MyDrive/DistilBERT_Hierarchical_emotion_detection.csv"
LOG_FILE_STEP2 = "/content/drive/MyDrive/DistilBERT_Hierarchical_intensity.csv"

with open(LOG_FILE_STEP1, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "epoch",
        "train_loss",
        "eval_loss",
        "f1_macro",
        "f1_micro"
    ])

with open(LOG_FILE_STEP2, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow([
        "epoch",
        "train_loss",
        "eval_loss",
        "f1_macro",
        "pearson"
    ])

print("Step 1 log file:", LOG_FILE_STEP1)
print("Step 2 log file:", LOG_FILE_STEP2)


In [ ]:

# =========================
# 2. SEED
# =========================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:

# =========================
# 3. LABELS
# =========================
EMOTIONS = ["anger", "fear", "joy", "sadness", "surprise"]
INTENSITY_COLUMNS = [f"{emotion}_intensity" for emotion in EMOTIONS]

# =========================
# 4. LOAD DATASET
# =========================
train_data = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="train")
val_data   = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="dev")
test_data  = load_dataset("brighter-dataset/BRIGHTER-emotion-intensities", "eng", split="test")

train_df = train_data.to_pandas()
val_df   = val_data.to_pandas()
test_df  = test_data.to_pandas()

print("Original sizes:")
print({
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df)
})


In [ ]:

# =========================
# 5. PREPARE TWO-STEP DATA
# =========================
def prepare_two_step_data(df):
    df = df.copy()

    if "disgust" in df.columns:
        df = df.drop(columns=["disgust"])

    # create intensity columns first
    for emotion in EMOTIONS:
        df[f"{emotion}_intensity"] = df[emotion].astype(int)

    # convert original emotion columns to presence/absence
    for emotion in EMOTIONS:
        df[emotion] = (df[f"{emotion}_intensity"] > 0).astype(int)

    return df

train_two = prepare_two_step_data(train_df)
val_two   = prepare_two_step_data(val_df)
test_two  = prepare_two_step_data(test_df)


In [ ]:

# =========================
# 6. MERGE + RESPLIT 70/20/10
# =========================
full_df = pd.concat([train_two, val_two, test_two], ignore_index=True)
full_df = full_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

n = len(full_df)
train_end = int(0.7 * n)
val_end = int(0.9 * n)

train_split = full_df[:train_end].reset_index(drop=True)
val_split   = full_df[train_end:val_end].reset_index(drop=True)
test_split  = full_df[val_end:].reset_index(drop=True)

print("New split sizes:")
print({
    "train": len(train_split),
    "val": len(val_split),
    "test": len(test_split)
})

# =========================
# 7. STEP 1 DATA
# =========================
train_step1_df = train_split[["text"] + EMOTIONS].copy()
val_step1_df   = val_split[["text"] + EMOTIONS].copy()
test_step1_df  = test_split[["text"] + EMOTIONS].copy()

# =========================
# 8. STEP 2 DATA
# =========================
train_step2_df = train_split[["text"] + INTENSITY_COLUMNS].copy()
val_step2_df   = val_split[["text"] + INTENSITY_COLUMNS].copy()
test_step2_df  = test_split[["text"] + INTENSITY_COLUMNS].copy()




In [ ]:
# =========================
# 9. CONVERT TO HF DATASETS
# =========================
train_step1_ds = Dataset.from_pandas(train_step1_df, preserve_index=False)
val_step1_ds   = Dataset.from_pandas(val_step1_df, preserve_index=False)
test_step1_ds  = Dataset.from_pandas(test_step1_df, preserve_index=False)

train_step2_ds = Dataset.from_pandas(train_step2_df, preserve_index=False)
val_step2_ds   = Dataset.from_pandas(val_step2_df, preserve_index=False)
test_step2_ds  = Dataset.from_pandas(test_step2_df, preserve_index=False)

# =========================
# 10. TOKENIZER
# =========================
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

train_step1_ds = train_step1_ds.map(tokenize_function, batched=True)
val_step1_ds   = val_step1_ds.map(tokenize_function, batched=True)
test_step1_ds  = test_step1_ds.map(tokenize_function, batched=True)

train_step2_ds = train_step2_ds.map(tokenize_function, batched=True)
val_step2_ds   = val_step2_ds.map(tokenize_function, batched=True)
test_step2_ds  = test_step2_ds.map(tokenize_function, batched=True)

# =========================
# 11. ADD LABELS
# =========================
def add_step1_labels(example):
    example["labels"] = [float(example[emotion]) for emotion in EMOTIONS]
    return example

def add_step2_labels(example):
    example["labels"] = [int(example[col]) for col in INTENSITY_COLUMNS]
    return example

train_step1_ds = train_step1_ds.map(add_step1_labels)
val_step1_ds   = val_step1_ds.map(add_step1_labels)
test_step1_ds  = test_step1_ds.map(add_step1_labels)

train_step2_ds = train_step2_ds.map(add_step2_labels)
val_step2_ds   = val_step2_ds.map(add_step2_labels)
test_step2_ds  = test_step2_ds.map(add_step2_labels)

# =========================
# 12. FORMAT
# =========================
train_step1_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_step1_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_step1_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

train_step2_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_step2_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_step2_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


In [ ]:

# =========================
# 13. STEP 1 MODEL
# =========================
model_step1 = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(EMOTIONS),
    problem_type="multi_label_classification"
).to(device)

# =========================
# 14. STEP 1 METRICS
# =========================
def compute_metrics_step1(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)
    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)

    return {
        "f1_macro": float(f1_macro),
        "f1_micro": float(f1_micro)
    }

# =========================
# 15. STEP 1 CALLBACK
# =========================
step1_epoch_list = []
step1_train_loss_list = []
step1_eval_loss_list = []
step1_f1_macro_list = []

class SaveMetricsCallbackStep1(TrainerCallback):
    def __init__(self, file_path):
        self.file_path = file_path
        self.current_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs and "eval_loss" not in logs:
            self.current_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is not None:
            epoch = float(metrics.get("epoch", state.epoch))
            train_loss = self.current_train_loss if self.current_train_loss is not None else ""
            eval_loss = float(metrics.get("eval_loss", 0.0))
            f1_macro = float(metrics.get("eval_f1_macro", 0.0))
            f1_micro = float(metrics.get("eval_f1_micro", 0.0))

            step1_epoch_list.append(epoch)
            step1_train_loss_list.append(train_loss)
            step1_eval_loss_list.append(eval_loss)
            step1_f1_macro_list.append(f1_macro)

            with open(self.file_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([
                    epoch,
                    train_loss,
                    eval_loss,
                    f1_macro,
                    f1_micro
                ])


In [ ]:

# =========================
# 16. STEP 1 TRAINING ARGS
# =========================
training_args_step1 = TrainingArguments(
    output_dir="/content/distilbert_step1_output",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

# =========================
# 17. STEP 1 TRAINER
# =========================
trainer_step1 = Trainer(
    model=model_step1,
    args=training_args_step1,
    train_dataset=train_step1_ds,
    eval_dataset=val_step1_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics_step1,
    callbacks=[SaveMetricsCallbackStep1(LOG_FILE_STEP1)]
)

# =========================
# 18. TRAIN STEP 1
# =========================
print("\nStarting Step 1: Emotion Detection")
start1 = time.time()
trainer_step1.train()
end1 = time.time()
print(f"Step 1 training time: {end1 - start1:.1f} seconds")


In [ ]:

# =========================
# 19. STEP 2 MODEL
# =========================
class DistilBertForMultiEmotionIntensity(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.num_emotions = len(EMOTIONS)   # 5
        self.num_classes = 4                # 0,1,2,3

        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased", config=config)
        self.dropout = nn.Dropout(config.seq_classif_dropout)
        self.classifier = nn.Linear(config.dim, self.num_emotions * self.num_classes)

    def forward(self, input_ids=None, attention_mask=None, labels=None, emotion_predictions=None, **kwargs):
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # DistilBERT has no pooler, so use first token representation
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)

        logits = self.classifier(cls_output)
        logits = logits.view(-1, self.num_emotions, self.num_classes)

        # Optional gating during forward if emotion_predictions is provided
        if emotion_predictions is not None:
            emotion_predictions = emotion_predictions.unsqueeze(-1).expand(-1, -1, self.num_classes)
            logits = logits * emotion_predictions

        loss = None
        if labels is not None:
            labels = labels.long()
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_classes), labels.view(-1))

        return {"loss": loss, "logits": logits}

config = DistilBertConfig.from_pretrained("distilbert-base-uncased")
model_step2 = DistilBertForMultiEmotionIntensity(config=config).to(device)

print("Custom DistilBERT model for Step 2 initialized successfully!")

# =========================
# 20. STEP 2 METRICS
# =========================
def compute_metrics_step2(eval_pred):
    logits, labels = eval_pred

    preds = np.argmax(logits, axis=-1)

    f1_macro = f1_score(
        labels.reshape(-1),
        preds.reshape(-1),
        average="macro",
        zero_division=0
    )

    true_flat = labels.reshape(-1)
    pred_flat = preds.reshape(-1)

    if np.std(true_flat) == 0 or np.std(pred_flat) == 0:
        pearson = 0.0
    else:
        pearson, _ = pearsonr(true_flat, pred_flat)
        if np.isnan(pearson):
            pearson = 0.0

    return {
        "f1_macro": float(f1_macro),
        "pearson": float(pearson)
    }


# =========================
# 21. STEP 2 CALLBACK
# =========================
step2_epoch_list = []
step2_train_loss_list = []
step2_eval_loss_list = []
step2_f1_macro_list = []
step2_pearson_list = []

class SaveMetricsCallbackStep2(TrainerCallback):
    def __init__(self, file_path):
        self.file_path = file_path
        self.current_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs and "eval_loss" not in logs:
            self.current_train_loss = float(logs["loss"])

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is not None:
            epoch = float(metrics.get("epoch", state.epoch))
            train_loss = self.current_train_loss if self.current_train_loss is not None else ""
            eval_loss = float(metrics.get("eval_loss", 0.0))
            f1_macro = float(metrics.get("eval_f1_macro", 0.0))
            pearson = float(metrics.get("eval_pearson", 0.0))

            step2_epoch_list.append(epoch)
            step2_train_loss_list.append(train_loss)
            step2_eval_loss_list.append(eval_loss)
            step2_f1_macro_list.append(f1_macro)
            step2_pearson_list.append(pearson)

            with open(self.file_path, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([
                    epoch,
                    train_loss,
                    eval_loss,
                    f1_macro,
                    pearson
                ])


In [ ]:

# =========================
# 22. STEP 2 TRAINING ARGS
# =========================
training_args_step2 = TrainingArguments(
    output_dir="/content/distilbert_step2_output",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

# =========================
# 23. STEP 2 TRAINER
# =========================
trainer_step2 = Trainer(
    model=model_step2,
    args=training_args_step2,
    train_dataset=train_step2_ds,
    eval_dataset=val_step2_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics_step2,
    callbacks=[SaveMetricsCallbackStep2(LOG_FILE_STEP2)]
)

# =========================
# 24. TRAIN STEP 2
# =========================
print("\nStarting Step 2: Intensity Classification")
start2 = time.time()
trainer_step2.train()
end2 = time.time()
print(f"Step 2 training time: {end2 - start2:.1f} seconds")


In [ ]:

# =========================
# 25. FINAL EVALUATION
# =========================
print("\nRunning final two-step evaluation on test set...")

# Step 1 prediction
pred_step1 = trainer_step1.predict(test_step1_ds)
logits_step1 = pred_step1.predictions
true_emotions = pred_step1.label_ids

probs_step1 = 1 / (1 + np.exp(-logits_step1))
pred_emotions = (probs_step1 >= 0.5).astype(int)

emotion_f1_macro = f1_score(true_emotions, pred_emotions, average="macro", zero_division=0)

# Step 2 prediction
pred_step2 = trainer_step2.predict(test_step2_ds)
logits_step2 = pred_step2.predictions
true_intensities = pred_step2.label_ids

pred_intensities = np.argmax(logits_step2, axis=-1)

# Gate Step 2 intensities using Step 1 emotion predictions
final_pred_intensities = pred_intensities * pred_emotions

intensity_f1_macro = f1_score(
    true_intensities.reshape(-1),
    final_pred_intensities.reshape(-1),
    average="macro",
    zero_division=0
)

overall_macro_f1 = intensity_f1_macro

true_flat = true_intensities.reshape(-1)
pred_flat = final_pred_intensities.reshape(-1)

if np.std(true_flat) == 0 or np.std(pred_flat) == 0:
    final_pearson = 0.0
else:
    final_pearson, _ = pearsonr(true_flat, pred_flat)
    if np.isnan(final_pearson):
        final_pearson = 0.0

print("\nFinal Two-Step Results")
print("----------------------")
print("Emotion Detection Macro F1:", emotion_f1_macro)
print("Intensity Prediction Macro F1:", intensity_f1_macro)
print("Overall Macro F1:", overall_macro_f1)
print("Final Pearson:", final_pearson)


In [ ]:

# =========================
# 26. PLOTS
# =========================
plt.figure(figsize=(8, 5))
plt.plot(step1_epoch_list, step1_train_loss_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Step 1 Training Loss vs Epoch")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(step1_epoch_list, step1_eval_loss_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Step 1 Validation Loss vs Epoch")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(step1_epoch_list, step1_f1_macro_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("F1 Macro")
plt.title("Step 1 F1 Macro vs Epoch")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(step2_epoch_list, step2_train_loss_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Step 2 Training Loss vs Epoch")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(step2_epoch_list, step2_eval_loss_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Validation Loss")
plt.title("Step 2 Validation Loss vs Epoch")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(step2_epoch_list, step2_f1_macro_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("F1 Macro")
plt.title("Step 2 F1 Macro vs Epoch")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(step2_epoch_list, step2_pearson_list, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Pearson")
plt.title("Step 2 Pearson vs Epoch")
plt.grid(True)
plt.show()